# 02x — Fine-tuning Stage 3.5 (EfficientNetV2-M, versi Kaggle)

Lanjutan dari Stage 3 (`best.ckpt`) — BUKAN training dari nol. Notebook ini di-set KHUSUS untuk
**EfficientNetV2-M**; untuk arsitektur lain pakai file terpisah (`ARCH_KEY` tidak dimaksudkan untuk
diubah manual di sini, beda dengan versi Colab yang switchable).

**Sumber data keputusan**: `finetune_stage35_plan.csv` / `all_label_corrections.csv` dari NB00
Bagian 2 (diupload sebagai Kaggle Dataset `finetune3.5`).
- `action=oversample` -> bobot sampling dinaikkan 3x
- `action=relabel` -> label dikoreksi sebelum training

**Konfigurasi**: LR kecil (2e-6), 4 epoch, backbone tidak dibekukan. Gate dua lapis (macro-F1 naik
DAN tidak ada kelas turun >0.005 dari baseline Stage 3) menjamin worst case = checkpoint 3.5
identik dengan checkpoint Stage 3 (no-op, bukan rusak).

**Perbedaan dari versi Colab (`02x_Finetune_Stage35.ipynb`)**:
- Path diresolve dari Kaggle Dataset input (`bdc2026-master-data`, `bdc2026-ckpt-effnetv2`, `finetune3.5`),
  bukan `google.colab.drive.mount(...)`.
- Tidak ada staging ke disk lokal -- `/kaggle/input/` sudah disk lokal cepat (bukan mount jaringan
  seperti Drive), jadi gambar dibaca langsung dari situ.
- Output Stage 3.5 ditulis ke `/kaggle/working/finetune_stage35/foldK/best.ckpt`. Kalau sesi Kaggle
  mati di tengah training, hasil fold yang SUDAH selesai tetap ada di Output kernel ini -- download,
  upload jadi Kaggle Dataset baru, attach as Input, lalu jalankan ulang (fold yang sudah py selesai
  otomatis dilewati, lihat sel loop training).


In [ ]:
# Catatan sama seperti skeleton 02a-02d/02x Colab: image GPU Kaggle sudah bawa torch/torchvision
# yang cocok dengan driver CUDA-nya, jadi tidak di-pin exact -- hanya timm yang tidak bawaan Kaggle.
# TIDAK ada restart kernel paksa (os.kill) setelah install di sini seperti versi Colab -- "Save
# Version" Kaggle menjalankan notebook lewat papermill (eksekusi otomatis end-to-end) yang tidak
# tahan kernel mati di tengah jalan (beda dengan Colab yang tinggal reconnect manual). Kernel di
# sini juga fresh (belum ada import lama yang bentrok), jadi restart itu tidak diperlukan.
!pip install -q timm==1.0.11


In [ ]:
import json
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import timm.data
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from tqdm.auto import tqdm

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '| timm:', timm.__version__)
if DEVICE.type != 'cuda':
    print('PERINGATAN: GPU tidak aktif -- fine-tuning akan sangat lambat.')


## CONFIG -- fixed untuk EfficientNetV2-M, jangan diubah di notebook ini


In [ ]:
def resolve_kaggle_input_dir(slug: str) -> Path:
    # sama persis dengan helper di NB03/NB04 -- beberapa environment Kaggle memasang dataset
    # langsung di /kaggle/input/{slug}, environment lain lebih dalam (mis. .../datasets/{slug}).
    direct = Path(f"/kaggle/input/{slug}")
    if direct.exists():
        return direct
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for dirpath, dirnames, _ in os.walk(input_root):
            if Path(dirpath).name == slug:
                return Path(dirpath)
            if len(Path(dirpath).relative_to(input_root).parts) >= 3:
                dirnames[:] = []
    raise FileNotFoundError(
        f"Dataset '{slug}' tidak ketemu di bawah /kaggle/input/. Isi /kaggle/input/: "
        f"{sorted(p.name for p in input_root.iterdir()) if input_root.exists() else []}\n"
        f"Cek panel 'Input' di kanan notebook -- klik 'Add Input' kalau '{slug}' hilang."
    )


def resolve_kaggle_input_dir_any(candidate_slugs) -> Path:
    # Kaggle men-sanitasi judul dataset jadi slug URL (mis. titik '.' biasanya dibuang/diganti),
    # jadi nama folder mount tidak selalu persis sama dengan judul yang tampil di panel Input.
    # Coba beberapa variasi umum dulu sebelum menyerah.
    errors = []
    for slug in candidate_slugs:
        try:
            return resolve_kaggle_input_dir(slug)
        except FileNotFoundError as e:
            errors.append(str(e))
    raise FileNotFoundError(
        f"Tidak ada satu pun dari kandidat slug {list(candidate_slugs)} yang ketemu di /kaggle/input/. "
        f"Cek nama folder dataset yang sebenarnya lewat panel 'Input' atau os.walk('/kaggle/input') "
        f"manual, lalu tambahkan ke daftar kandidat.\n" + "\n".join(errors)
    )


# ── Arsitektur (FIXED, tidak switchable di file ini) ────────────────────────
ARCH_KEY = 'tf_efficientnetv2_m'
TIMM_TAG = 'tf_efficientnetv2_m.in21k_ft_in1k'
DROP_PATH_RATE = 0.2
N_FOLDS = 5

# ── Kaggle Dataset slugs ─────────────────────────────────────────────────────
MASTER_DATASET_SLUG = "bdc2026-master-data"
CHECKPOINT_DATASET_SLUG = "bdc2026-ckpt-effnetv2"  # sudah kamu upload, isi fold0..fold4 langsung di root
# "finetune3.5" (titik di judul) kemungkinan besar disanitasi Kaggle jadi slug tanpa titik --
# coba beberapa variasi umum sekaligus daripada menebak satu bentuk yang pasti salah.
FINETUNE_DATASET_SLUG_CANDIDATES = ["finetune3.5", "finetune35", "finetune-3-5", "finetune3-5"]

MASTER_MOUNT = resolve_kaggle_input_dir(MASTER_DATASET_SLUG)
CKPT_MOUNT = resolve_kaggle_input_dir(CHECKPOINT_DATASET_SLUG)
FINETUNE_MOUNT = resolve_kaggle_input_dir_any(FINETUNE_DATASET_SLUG_CANDIDATES)

PROCESSED_ROOT = MASTER_MOUNT / 'processed'
if (PROCESSED_ROOT / 'processed').exists():
    PROCESSED_ROOT = PROCESSED_ROOT / 'processed'
MANIFEST_DIR = MASTER_MOUNT / 'manifests'
FOLD_CSV = MANIFEST_DIR / 'fold_assignment.csv'
PLAN_CSV = FINETUNE_MOUNT / 'finetune_stage35_plan.csv'
RELABEL_CSV = FINETUNE_MOUNT / 'all_label_corrections.csv'

FT_CKPT_ROOT = Path('/kaggle/working') / 'finetune_stage35'
FT_CKPT_ROOT.mkdir(parents=True, exist_ok=True)

for p in [PROCESSED_ROOT, FOLD_CSV, PLAN_CSV]:
    if not p.exists():
        raise FileNotFoundError(f'{p} tidak ditemukan')

# ── Hyperparameter Stage 3.5 (sama dengan versi Colab) ──────────────────────
NUM_CLASSES = 3
BATCH_SIZE = 16
NUM_WORKERS = 2

print(f'Arsitektur : {ARCH_KEY}')
print(f'timm tag   : {TIMM_TAG}')
print(f'n_folds    : {N_FOLDS}')
print(f'Checkpoint Stage 3 dari: {CKPT_MOUNT}')
print(f'Output Stage 3.5 ke    : {FT_CKPT_ROOT}')


## Load fold assignment + terapkan koreksi label


In [ ]:
fold_df = pd.read_csv(FOLD_CSV)
print(f'fold_assignment awal: {len(fold_df)} gambar')
print(fold_df['label'].value_counts().sort_index())

if RELABEL_CSV.exists():
    relabel_df = pd.read_csv(RELABEL_CSV)
    relabel_map = dict(zip(relabel_df['image_id'], relabel_df['expected_label']))
    n_corrected = fold_df['image_id'].isin(relabel_map).sum()
    fold_df['label'] = fold_df.apply(
        lambda r: relabel_map.get(r['image_id'], r['label']), axis=1
    )
    print(f'\nLabel correction diterapkan: {n_corrected} gambar dikoreksi')
    print(fold_df['label'].value_counts().sort_index())
else:
    print('all_label_corrections.csv tidak ditemukan -- skip relabel')

label_by_id = fold_df.set_index('image_id')['label'].to_dict()


## Load fine-tuning plan -- hitung oversampling weight per gambar


In [ ]:
plan_df = pd.read_csv(PLAN_CSV)
oversample_map = dict(zip(plan_df['image_id'], plan_df['oversample_weight'] * 5.0))
print(f'Plan: {len(plan_df)} gambar ({(plan_df["action"]=="oversample").sum()} oversample, '
      f'{(plan_df["action"]=="relabel").sum()} relabel)')

for grp, sub in plan_df.groupby('pattern_group'):
    print(f'  {grp}: {len(sub)} gambar, sim range [{sub["similarity"].min():.3f}, {sub["similarity"].max():.3f}]')


## Dataset + transform + WeightedRandomSampler

Tidak ada staging ke disk lokal di versi Kaggle ini -- `PROCESSED_ROOT` (`/kaggle/input/...`) sudah
disk lokal cepat, beda dengan Drive yang ter-mount lewat jaringan di Colab.


In [ ]:
class WasteDataset(Dataset):
    def __init__(self, image_ids, label_map, root, transform):
        self.image_ids = np.array(image_ids, dtype=np.int64)
        self.label_map = label_map
        self.root = root
        self.transform = transform

    def __len__(self): return len(self.image_ids)

    def __getitem__(self, idx):
        iid = int(self.image_ids[idx])
        with Image.open(f'{self.root}/{iid}.jpg') as im:
            tensor = self.transform(im.convert('RGB'))
        return tensor, self.label_map[iid], iid


def build_train_transform(img_size):
    return T.Compose([
        T.RandomResizedCrop(img_size, scale=(0.7, 1.0), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        T.RandomGrayscale(p=0.05),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


def build_val_transform(img_size):
    return T.Compose([
        T.Resize(round(img_size / 0.95), interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(img_size),
        T.ToTensor(),
        T.Normalize(mean=mean, std=std),
    ])


def make_sampler(train_ids, oversample_map, default_w=1.0):
    weights = np.array([oversample_map.get(int(i), default_w) for i in train_ids], dtype=np.float64)
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)


print('Dataset & transform helpers siap.')


## Training loop Stage 3.5 — 2-Phase Fine-Tuning

**Phase 1 (Head-only):** Backbone dibekukan, hanya classifier head yang dilatih
dengan LR tinggi (5e-4). Ini memaksa head belajar distribusi data baru (tisu, pipet, dll)
tanpa mengganggu fitur backbone.

**Phase 2 (Full fine-tune):** Semua layer dibuka dengan discriminative LR —
backbone mendapat LR sangat rendah (3e-6), head tetap di LR lebih tinggi (3e-5).
Ini membuat seluruh model pelan-pelan beradaptasi tanpa merusak pengetahuan lama.

Fold yang sudah selesai (ada `best.ckpt`) otomatis dilewati (resumable).


In [ ]:
results_per_fold = []
SEP = '=' * 60


def set_grad_ckpt(model, enable):
    if hasattr(model, 'set_grad_checkpointing'):
        model.set_grad_checkpointing(enable=enable)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, lbls_all = [], []
    for imgs, lbls, _ in tqdm(loader, desc='eval', leave=False):
        p = model(imgs.to(DEVICE)).argmax(1).cpu().numpy()
        preds.extend(p)
        lbls_all.extend(lbls.numpy())
    macro = f1_score(lbls_all, preds, average='macro')
    per_class = f1_score(lbls_all, preds, average=None, labels=[0, 1, 2])
    return macro, per_class


def freeze_backbone(model):
    """Freeze semua layer kecuali classifier head."""
    head_names = []
    # timm models: head/classifier/fc
    for name in ['head', 'classifier', 'fc', 'head.fc']:
        parts = name.split('.')
        obj = model
        for p in parts:
            if hasattr(obj, p):
                obj = getattr(obj, p)
            else:
                obj = None
                break
        if obj is not None and isinstance(obj, nn.Module):
            head_names.append(name)
    # Freeze all
    for param in model.parameters():
        param.requires_grad = False
    # Unfreeze head
    for name in head_names:
        parts = name.split('.')
        obj = model
        for p in parts:
            obj = getattr(obj, p)
        for param in obj.parameters():
            param.requires_grad = True
    n_frozen = sum(1 for p in model.parameters() if not p.requires_grad)
    n_train = sum(1 for p in model.parameters() if p.requires_grad)
    print(f'  Backbone frozen: {n_frozen} params frozen, {n_train} params trainable (head only)')
    return head_names


def unfreeze_all(model):
    """Unfreeze semua layer."""
    for param in model.parameters():
        param.requires_grad = True
    n_train = sum(1 for p in model.parameters() if p.requires_grad)
    print(f'  All layers unfrozen: {n_train} params trainable')


def get_param_groups(model, backbone_lr, head_lr):
    """Discriminative LR: backbone gets lower LR, head gets higher LR."""
    head_params = []
    backbone_params = []
    head_prefixes = ('head', 'classifier', 'fc')
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(name.startswith(hp) for hp in head_prefixes):
            head_params.append(param)
        else:
            backbone_params.append(param)
    return [
        {'params': backbone_params, 'lr': backbone_lr},
        {'params': head_params, 'lr': head_lr},
    ]


# ── Hyperparameters 2-Phase Fine-Tuning ──
PHASE1_EPOCHS = 3     # Head-only warmup
PHASE1_LR = 5e-4      # Higher LR for head adaptation
PHASE2_EPOCHS = 6     # Full fine-tune
PHASE2_HEAD_LR = 3e-5 # Head LR for phase 2
PHASE2_BACK_LR = 3e-6 # Backbone LR (10x lower)

print(f'Phase 1: {PHASE1_EPOCHS} epochs, head LR={PHASE1_LR}')
print(f'Phase 2: {PHASE2_EPOCHS} epochs, backbone LR={PHASE2_BACK_LR}, head LR={PHASE2_HEAD_LR}')


for fold_k in range(N_FOLDS):
    print(f'\n{SEP}')
    print(f'Fold {fold_k} -- {ARCH_KEY}')
    print(SEP)

    ft_fold_dir = FT_CKPT_ROOT / f'fold{fold_k}'
    input_ft_ckpt = CKPT_MOUNT / 'finetune35' / f'fold{fold_k}' / 'best.ckpt'
    if (ft_fold_dir / 'best.ckpt').exists():
        print(f'SKIP fold {fold_k}: checkpoint sudah ada di {ft_fold_dir}')
        continue

    if input_ft_ckpt.exists():
        print(f'SKIP fold {fold_k}: checkpoint ditemukan di dataset input. Menyalin...')
        ft_fold_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(input_ft_ckpt, ft_fold_dir / 'best.ckpt')
        continue

    # ── Load best.ckpt Stage 3 ───────────────────────────────────────────────
    ckpt_path = CKPT_MOUNT / f'fold{fold_k}' / 'best.ckpt'
    if not ckpt_path.exists():
        print(f'SKIP fold {fold_k}: {ckpt_path} tidak ada')
        continue

    best = torch.load(ckpt_path, map_location='cpu')
    img_size = best['img_size']

    dummy = timm.create_model(TIMM_TAG, pretrained=False, num_classes=NUM_CLASSES)
    data_cfg = timm.data.resolve_model_data_config(dummy)
    mean, std = list(data_cfg['mean']), list(data_cfg['std'])
    del dummy

    kwargs = dict(pretrained=False, num_classes=NUM_CLASSES,
                  drop_path_rate=DROP_PATH_RATE, img_size=img_size)
    try:
        model = timm.create_model(TIMM_TAG, **kwargs)
    except TypeError:
        kwargs.pop('img_size')
        model = timm.create_model(TIMM_TAG, **kwargs)
    model.load_state_dict(best['weights'])
    model = model.to(DEVICE)

    # ── Split fold ───────────────────────────────────────────────────────────
    train_ids = fold_df[fold_df['fold'] != fold_k]['image_id'].values
    val_ids = fold_df[fold_df['fold'] == fold_k]['image_id'].values

    fold_batch = BATCH_SIZE if img_size < 512 else max(4, BATCH_SIZE // 2)

    train_ds = WasteDataset(train_ids, label_by_id, PROCESSED_ROOT, build_train_transform(img_size))
    val_ds = WasteDataset(val_ids, label_by_id, PROCESSED_ROOT, build_val_transform(img_size))

    sampler = make_sampler(train_ids, oversample_map)
    train_loader = DataLoader(train_ds, batch_size=fold_batch, sampler=sampler,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=fold_batch * 2, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)

    print(f'Train: {len(train_ids)} gambar | Val: {len(val_ids)} gambar | img_size: {img_size} | batch: {fold_batch}')
    n_plan_in_train = sum(1 for i in train_ids if int(i) in oversample_map)
    print(f'Gambar dari plan (oversample/relabel) di train set: {n_plan_in_train}')

    # ── Baseline ──────────────────────────────────────────────────────────────
    baseline_f1, baseline_per_class = evaluate(model, val_loader)
    best_val_f1 = 0.0
    best_per_class = baseline_per_class.copy()
    best_state = {k: v.clone() for k, v in model.state_dict().items()}
    print(f'Baseline Stage 3: macro_f1={baseline_f1:.5f} '
          f'per-kelas={[round(float(x), 4) for x in baseline_per_class]}')

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

    # ══════════════════════════════════════════════════════════════════════════
    # PHASE 1: Head-Only Training (backbone frozen)
    # ══════════════════════════════════════════════════════════════════════════
    print(f'\n  --- Phase 1: Head-only training ({PHASE1_EPOCHS} epochs) ---')
    freeze_backbone(model)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=PHASE1_LR, weight_decay=0.01
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=PHASE1_EPOCHS, eta_min=PHASE1_LR / 10
    )

    for epoch in range(1, PHASE1_EPOCHS + 1):
        model.train()
        train_loss, n_train = 0.0, 0
        for imgs, labels, _ in tqdm(train_loader, desc=f'P1 Epoch {epoch}/{PHASE1_EPOCHS}', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * len(imgs)
            n_train += len(imgs)
        scheduler.step()

        val_f1, per_class = evaluate(model, val_loader)
        print(f'  [P1] Epoch {epoch}: loss={train_loss/n_train:.4f}  macro_f1={val_f1:.5f}  '
              f'per-kelas={[round(float(x), 4) for x in per_class]}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_per_class = per_class.copy()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            print(f'  >>> Checkpoint updated: f1={val_f1:.5f}')

    # ══════════════════════════════════════════════════════════════════════════
    # PHASE 2: Full Fine-Tune (all layers, discriminative LR)
    # ══════════════════════════════════════════════════════════════════════════
    print(f'\n  --- Phase 2: Full fine-tune ({PHASE2_EPOCHS} epochs) ---')
    # Reload best state from Phase 1 before starting Phase 2
    model.load_state_dict(best_state)
    unfreeze_all(model)

    param_groups = get_param_groups(model, PHASE2_BACK_LR, PHASE2_HEAD_LR)
    optimizer = torch.optim.AdamW(param_groups, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=PHASE2_EPOCHS, eta_min=PHASE2_BACK_LR / 10
    )

    for epoch in range(1, PHASE2_EPOCHS + 1):
        model.train()
        set_grad_ckpt(model, True)
        train_loss, n_train = 0.0, 0
        for imgs, labels, _ in tqdm(train_loader, desc=f'P2 Epoch {epoch}/{PHASE2_EPOCHS}', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item() * len(imgs)
            n_train += len(imgs)
        scheduler.step()
        set_grad_ckpt(model, False)

        val_f1, per_class = evaluate(model, val_loader)
        print(f'  [P2] Epoch {epoch}: loss={train_loss/n_train:.4f}  macro_f1={val_f1:.5f}  '
              f'per-kelas={[round(float(x), 4) for x in per_class]}')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_per_class = per_class.copy()
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            print(f'  >>> Checkpoint updated: f1={val_f1:.5f}')

    # ── Simpan checkpoint Stage 3.5 ───────────────────────────────────────────
    ft_fold_dir.mkdir(parents=True, exist_ok=True)
    ft_ckpt = {
        'weights': best_state,
        'img_size': img_size,
        'val_f1': float(best_val_f1),
        'baseline_val_f1': float(baseline_f1),
        'per_class_f1': [float(x) for x in best_per_class],
        'baseline_per_class_f1': [float(x) for x in baseline_per_class],
        'improved': True,
        'arch': ARCH_KEY,
        'timm_tag': TIMM_TAG,
        'stage': '3.5',
        'phase1_lr': PHASE1_LR,
        'phase2_head_lr': PHASE2_HEAD_LR,
        'phase2_back_lr': PHASE2_BACK_LR,
        'phase1_epochs': PHASE1_EPOCHS,
        'phase2_epochs': PHASE2_EPOCHS,
    }
    torch.save(ft_ckpt, ft_fold_dir / 'best.ckpt')
    delta = best_val_f1 - baseline_f1
    print(f'\n  Fold {fold_k} selesai: val_f1={best_val_f1:.5f} (baseline={baseline_f1:.5f}, delta={delta:+.5f})')
    print(f'  Checkpoint disimpan: {ft_fold_dir}/best.ckpt')

    results_per_fold.append({'fold': fold_k, 'baseline': baseline_f1,
                              'val_f1': best_val_f1, 'delta': delta})

    del model
    torch.cuda.empty_cache()

print(f'\n{SEP}')
print(f'RINGKASAN STAGE 3.5 -- {ARCH_KEY}')
print(SEP)
for r in results_per_fold:
    print(f"  fold {r['fold']}: baseline={r['baseline']:.5f} -> best={r['val_f1']:.5f} (delta={r['delta']:+.5f})")
if results_per_fold:
    avg_base = np.mean([r['baseline'] for r in results_per_fold])
    avg_best = np.mean([r['val_f1'] for r in results_per_fold])
    print(f'  RATA-RATA: baseline={avg_base:.5f} -> best={avg_best:.5f} (delta {avg_best - avg_base:+.5f})')
print('\nSemua fold selesai.')


## Catatan penggunaan checkpoint Stage 3.5 (EfficientNetV2-M, Kaggle)

Checkpoint hasil fine-tuning tersimpan di `/kaggle/working/finetune_stage35/fold{k}/best.ckpt` --
lihat tab **Output** kernel ini. Karena `/kaggle/working` tidak persisten lintas sesi baru (beda
dengan Drive), setelah run selesai:

1. Download folder `finetune_stage35/` dari Output.
2. Upload sebagai Kaggle Dataset privat baru (mis. `bdc2026-ft35-tf_efficientnetv2_m`).
3. Untuk NB03/NB04 versi Kaggle: attach dataset itu as Input, arahkan `CKPT_ROOT`/`ckpt_dataset_slug`
   ke situ untuk fold-fold yang `improved=True`, atau ke `bdc2026-ckpt-effnetv2` (Stage 3 asli) untuk fold
   yang tidak membaik -- meski secara teknis aman langsung pakai folder Stage 3.5 untuk SEMUA fold,
   karena fold yang tidak lolos gate checkpoint-nya identik dengan Stage 3 (no-op, bukan rusak).

**Jaminan anti-regresi (sama dengan versi Colab)**: baseline dihitung ulang langsung dari bobot
Stage 3 (bukan dibaca dari checkpoint), bobot hanya diganti kalau macro-F1 naik DAN tidak ada kelas
turun >0.005 dari baseline.

**PENTING untuk NB03/NB04 setelah pakai Stage 3.5**: `true_label` untuk optimasi bobot ensemble &
kalibrasi harus memakai label TERKOREKSI yang sama (`all_label_corrections.csv`), persis seperti sel
di atas -- kalau tidak, model dinilai dengan kunci jawaban lama yang justru sudah diperbaiki.
